### Forecasting Human Emotions: Modeling Multiple Time Series with LSTMs and VAR

This notebook aims to address a challenging task: forecasting human emotions over time using time-series models. The dataset consists of **100 multiple time series** recordings, each capturing emotional dynamics over **1 to 10 minutes** at **0.1-second intervals**. Each time series is represented as a **600x7 matrix** (assuming a 1-minute length), where **600 timesteps** are recorded, and each timestep contains **7 features** representing different emotions as percentage values (e.g., `t=4.2s`, `[0.5, 0.2, 0.1, 0, 0.2, 0, 0]` means **50% happy**, **20% sad**, etc.).

The ultimate objective is to design a **machine learning model** capable of training on these time series to **forecast emotional states** for a **future time window**. In this notebook, we aim to determine the optimal number of lags to give the model and the ideal forecast length through experimentation. For ease of explanation, we will use **600 timesteps** to forecast the average emotion for the next **100 timesteps**, but these values will eventually be determined based on model performance.

### Objectives of the Notebook:

1. **Explore and Compare Modeling Techniques:**
   - We will test multiple **Long Short-Term Memory (LSTM)** architectures to handle the complex temporal relationships inherent in emotional data.
   - Additionally, we will explore models like **CNN-LSTM** and **ConvLSTM**, which may better capture local temporal features.
   - We will also compare these neural models to a more classical approach, the **Vector AutoRegression (VAR)** model, with both recursive and potentially direct multi-step forecasting.

2. **Design a Robust Experimentation Workflow:**
   - Split the dataset into **training** and **testing** subsets using non-overlapping segments of each time series (an **80-20 split**), ensuring there is no data leakage between sets.
   - Use a range of architectures, both with standard LSTMs and variations such as **Augmented LSTMs**, to evaluate their forecasting capability.
   - Experiment with **recursive prediction** and **direct multi-step prediction** for both LSTMs and VAR models to determine the best methodology for this task.

3. **Modeling and Tuning Variables:**
   - As mentioned, the number of timesteps to forecast (i.e., the forecast length) and the number of past observations (i.e., lags) to use as input features will be treated as hyperparameters. We will conduct experiments to determine which combination yields the best performance for emotional forecasting.
   - Initially, for demonstration purposes, we will use **600 timesteps** as the lookback window and forecast the mean emotion vector for next **100 timesteps**.

### Outline of the Notebook:

1. **Data Preprocessing:**
   - Prepare the data by splitting each time series into segments of **training** and **testing** sets, ensuring **no overlap** exists between them to prevent leakage.
   - Explore several data scaling methods to determine which yields the best results for time-series models.

2. **Exploration of LSTM Architectures:**
   - We will evaluate **multiple LSTM variations**, including:
     - **Basic LSTM models**
     - **Activated LSTM models** with ReLU activations
     - **Multistep LSTMs**
     - **Augmented LSTMs**, which concatenate additional features like recent timesteps to enhance context awareness
     - **CNN-LSTM** and **ConvLSTM** models to capture spatio-temporal dynamics

3. **VAR Modeling:**
   - Apply a **Vector AutoRegression (VAR)** model for comparison.
   - Implement both **recursive prediction** and potentially **direct multi-step forecasting** approaches to analyze their efficacy.

4. **Performance Metrics and Evaluation:**
   - Compare models using standard metrics such as **Mean Squared Error (MSE)** and evaluate on both training and testing sets.
   - Visualize **training and validation loss curves**, and plot **predicted vs. actual emotion vectors** for both training and test data.

5. **Results Analysis:**
   - Summarize the findings from testing different models.
   - Provide insights into which models and hyperparameter choices performed best and why.

### Expected Outcomes:

- A deeper understanding of which machine learning techniques work best for forecasting **multi-variate emotional time series**.
- Insights into the challenges and limitations of LSTMs, CNN-LSTMs, ConvLSTMs, and VAR models for this task.
- Identification of the optimal **number of lags** and **forecast window** length for this dataset.

In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import sys
import os
sys.path.append('/Users/arnav/Google Drive/Cornell/Coding Projects/emili_TimeSeriesPredictor') #going up several files until emili_TimeSeriesPredictor
from time_series_predictor.Data.emotionFeatureExtractor import emotionFeatureExtractor
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from statsmodels.tsa.stattools import adfuller, acf
from time_series_predictor.Models.EmotionAverageModels import *

### Loading data

In [ ]:
# Loading data
extractor = emotionFeatureExtractor()
all_data = extractor.read_emotion_logs()
#obtains a train val and testing split
#should be such that all the data is saved in the same split random state

emotionMap = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Sadness', 'Surprise', 'Neutral']
#data imputation
#uses 'ewmainterp' method for resampling - determined it was the best method
resampled_data = [extractor.resample_data(file_data, 'ewmainterp') for file_data in all_data]

In [ ]:
plt.close(1)
#plotting one of the timeseries
def plot_emotion_data(all_series, iFile):
    #input is a single timeseries of size (length,7)
    data = all_series[iFile]
    t = np.arange(0,0.1*np.shape(data)[0],0.1)[:np.shape(data)[0]] #added [:np.shape(data)[0]] due to floating point error (e.g. 3102*0.1 == 310.2000005)
    nFeatures = np.shape(data)[1]

    fig, axes = plt.subplots(nFeatures, 1, figsize=(10, 1 * nFeatures), sharex=True, sharey = True)
    fig.suptitle(f'Features vs Time, file {iFile}')

    for iFeature in range(nFeatures):
        axes[iFeature].plot(t, data[:, iFeature], label='Data', marker = 'o', linestyle = '-', markersize=3)

    #label subplots
    for i in range(nFeatures):       
        axes[i].set_ylabel(f'{emotionMap[i]}')
        axes[i].grid(True)

    axes[-1].set_xlabel('Time (s)')
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)  # Adjust title position
    axes[-1].set_xlim(0, 100)
    axes[-1].set_ylim(0, 1)
    plt.show()

#Finding which sample has the most variance

var_per_sample = np.array([np.sum(np.var(resampled_data[i],axis=0)) for i in range(len(resampled_data))])
max_var_sample_idx = np.argmax(var_per_sample)

#plotting

sampleToPlot = 30 # CHANGE TO SEE DIFFERENT SAMPLES
plot_emotion_data(resampled_data,sampleToPlot)
print(plt.gcf().number)

The plot above shows the probability of each emotion at each timestep for a specific sample. A 'sample' signifies one time series of 1-10 minutes long that records the emotions of the user while they are chatting with a LM/listening to music/watching a movie etc. We allow the samples to not be based solely on chats with language models for ease of data collection, and it may also help the model generalize to have a better understanding of human emotion in general – not just limited to during chats with language models.

For sample 0 (shown above if you select it), the majority emotion is neutral, with some very small spikes/increases in some other emotions. However, looking at sample 30 (statistically found to be the one with the most variance), we can see that when there is an increase in one emotion, there is a decrease in the other – the definition of compositional data. Additionally, we can see that while there are increases in some emotions, the baseline average is typically returned to quite quickly, suggesting the data is stationary. We confirm this hypothesis below

### Exploring Statistical Properties of Emotion Data

#### Stationarity testing

In [ ]:
def analyze_stationarity_multivariate(all_series):
    """
    Apply stationarity tests to each feature in multivariate time series data.

    Parameters:
    - all_series: List of multivariate time series, each of shape (timesteps, features).

    Returns:
    - results: Dictionary with feature-wise results.
    """
    nFeatures = all_series[0].shape[1]
    results = {f'Feature_{i}': [] for i in range(nFeatures)}

    for series in all_series:
        for feature_idx in range(nFeatures):
            feature_data = series[:, feature_idx]
            adf_test = adfuller(feature_data, autolag='AIC')
            p_value = adf_test[1]
            results[f'Feature_{feature_idx}'].append(p_value)

    # Summarize results
    for feature, p_values in results.items():
        stationary_count = sum(p <= 0.05 for p in p_values)
        print(f"{feature}: {stationary_count}/{len(p_values)} series are stationary")
    
    return results

adf_results = analyze_stationarity_multivariate(resampled_data)

#### Autocorrelation for the features

In [ ]:
plt.close(2)
def plot_acf_multivariate(all_series, max_lag=600):
    """
    Plot autocorrelation for each feature in multivariate time series.

    Parameters:
    - all_series: List of multivariate time series, each of shape (timesteps, features).
    - max_lag: Maximum lag for autocorrelation.

    Returns:
    - None
    """
    n_features = all_series[0].shape[1]
    aggregated_acf = {f'Feature_{i}': np.zeros(max_lag + 1) for i in range(n_features)}

    nSeries = 0
    aggregated_length = 0

    for i,series in enumerate(all_series):
        if np.shape(series)[0] < 600:
            continue
        nSeries += 1
        for feature_idx in range(n_features):
            feature_data = series[:, feature_idx]
            acf_values = acf(feature_data, nlags=max_lag)
            aggregated_acf[f'Feature_{feature_idx}'] += acf_values
        aggregated_length += np.shape(series)[0] 

    # Average ACF across all series for each feature
    for feature in aggregated_acf:
        aggregated_acf[feature] /= nSeries

    conf_interval = 1.96 / np.sqrt(aggregated_length/nSeries) #95%

    # Plot averaged ACF for each feature
    plt.figure(figsize=(10, 6))
    for feature_idx, (feature, acf_values) in enumerate(aggregated_acf.items()):
        plt.plot(range(max_lag + 1), acf_values, label=f"{emotionMap[feature_idx]}",  markersize=3)
    # Add confidence interval dashed lines
        
    plt.axhline(y=conf_interval, color='black', linestyle='--', linewidth=1, label='95% Confidence Interval')
    plt.axhline(y=-conf_interval, color='black', linestyle='--', linewidth=1)

    plt.title("Average ACF for Each Feature Across Time Series")
    plt.xlabel("Lag")
    plt.ylabel("Autocorrelation")
    plt.ylim([-0.5,1])
    plt.legend()
    plt.grid(True)
    plt.show()

iFile = 12
plot_acf_multivariate(resampled_data)

## Training and Testing Split

### Methodology

**Goal**: Use p lags to predict the mean emotion vector for the next h time steps (h is the forecasting horizon). I.e. Use $\{\vec{x}_{t-p},\vec{x}_{t-p+1},...,\vec{x}_{t-1}\}$ to predict $\vec{y_t} = 1/h\sum_{i=t}^{t+h-1}{\vec{x_i}}$

To split the data, we have a couple of methods to turn this into a supervised learning problem.
1. Use the whole time series (minus the last h seconds) to predict the average emotion vector over the next h timesteps. The issue with this is that the samples vary in length, so our model will need to be able to take varied input lengths to forecast the mean.
2. Use a sliding window with (stride as a parameter to determine degree of overlap). This means that each sample will be partitioned into $\left\lfloor{\frac{sample_{}length}{p+h}}\right\rfloor$ sets (segments). When doing the training and testing split, we need to ensure that there are no samples that have one segment in the training set and one segment in the testing set as this would cause leakage into the test set. Therefore, when doing the 80/20 training and testing split, we split by the overall time series sample, not by the segments, and we split based on the length of the time series.

In [ ]:
### Train Test Split Parameters ###
p = 600 # number of timesteps to use to predict h steps into the future (600 = 60s)
h = 100 # number of timesteps in the future to average over for prediction
stride = p # setting equal to p so that there is no overlap between x data points
split = [0.8,0.2] # train test split

extractor.nLags = p
extractor.nForecastHorizon = h
extractor.stride = stride

In [ ]:
train_data,test_data = extractor.train_test_split(data=resampled_data, split=split,random_state=5)
xTr, yTr_raw = extractor.segment_data(data=train_data,resample_method='ewmainterp')
xTe, yTe_raw = extractor.segment_data(data=test_data,resample_method='ewmainterp')

# making the predictions the mean over h seconds
yTr = 1/h*np.sum(yTr_raw,axis=1,keepdims=True)
yTe = 1/h*np.sum(yTe_raw,axis=1,keepdims=True)

In [ ]:
plt.close(3)
#input is a single timeseries of size (length,7)
nFeatures = 7
nSample = 12
x = xTr[nSample]
y = yTr_raw[nSample]
y_mean=np.broadcast_to(yTr[nSample],(h,nFeatures))
t = np.arange(0,0.1*(p+h),0.1)[:(p+h)] #added [:np.shape(data)[0]] due to floating point error (e.g. 3102*0.1 == 310.2000005)


feature_data_fig, feature_data_axes = plt.subplots(nFeatures, 1, figsize=(10, 1 * nFeatures), sharex=True, sharey = True)
feature_data_fig.suptitle(f'Features vs Time, file {nSample}')

for iFeature in range(nFeatures):
    feature_data_axes[iFeature].plot(t[:p], x[:, iFeature], label='x', marker = 'o', linestyle = '-', markersize=3, color='red')
    feature_data_axes[iFeature].plot(t[p:p+h], y[:, iFeature], label='Actual Emotion', marker = 'o', linestyle = '-', markersize=3, color='black')
    feature_data_axes[iFeature].plot(t[p:p+h], y_mean[:, iFeature], label='y', linestyle = '-', linewidth=3, color='green')


#label subplots
for i in range(nFeatures):       
    feature_data_axes[i].set_ylabel(f'{emotionMap[i]}')
    feature_data_axes[i].grid(True)

feature_data_axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.subplots_adjust(top=0.95)  # Adjust title position
feature_data_axes[-1].set_xlim(0, 100)
feature_data_axes[-1].set_ylim(0, 1)
plt.legend()
plt.show()

## Emotion Prediction Models

### Training Function

In [ ]:
def train(model, xTrain, yTrain, xVal=None, yVal=None, epochs=10, batch_size=1, lr=0.001, shuffle=True):
    """
    General training function for PyTorch models.
    
    Parameters:
    - model (nn.Module): The PyTorch model to be trained.
    - xTrain (numpy.ndarray): Training data of shape (num_samples, timesteps, features).
    - yTrain (numpy.ndarray): Training labels of shape (num_samples, target_dim).
    - xVal (numpy.ndarray, optional): Validation data of shape (num_samples, timesteps, features).
    - yVal (numpy.ndarray, optional): Validation labels of shape (num_samples, target_dim).
    - epochs (int, optional): Number of epochs to train (default is 10).
    - batch_size (int, optional): Batch size (default is 32).
    - lr (float, optional): Learning rate for the optimizer (default is 0.001).
    - device (str, optional): Device to use for training ('cpu' or 'cuda'). If None, defaults to 'cpu'.
    
    Returns:
    - epoch_train_loss (list): History of training loss per epoch.
    - epoch_val_loss (list): History of validation loss per epoch (if validation data is provided).
    """

    # Convert numpy arrays to PyTorch tensors and move them to the device
    xTrain = torch.from_numpy(xTrain).float()
    yTrain = torch.from_numpy(yTrain).float()
    
    if xVal is not None and yVal is not None:
        xVal = torch.from_numpy(xVal).float()
        yVal = torch.from_numpy(yVal).float()

    # Define loss function and optimizer
    criterion = nn.MSELoss(reduction='none')  # Can be changed based on the task (e.g., CrossEntropyLoss for classification)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # History to store loss per epoch
    epoch_val_loss = []
    epoch_train_loss = []

    # Training loop
    for epoch in range(epochs):
        train_loss_per_batch = [] # TODO: determine what this represents
        model.train()  # Set model to training mode
        epoch_loss = 0.0
        
        if shuffle:
            # Shuffle the data at the beginning of each epoch
            permutation = torch.randperm(xTrain.size(0))
        else:
            # If no shuffling, maintain original order
            permutation = torch.arange(xTrain.size(0))

        # Process each batch
        for i in range(0, len(xTrain), batch_size):
            indices = permutation[i:i+batch_size]
            x_batch = xTrain[indices]
            y_batch = yTrain[indices]
            
            optimizer.zero_grad()  # Zero the gradients

            # Forward pass
            outputs = model(x_batch)
            assert(outputs.shape == y_batch.shape) #ensuring input and output sizes always match
            loss = torch.sum(criterion(outputs, y_batch))

            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            train_loss_per_batch.append(epoch_loss/(i+1)) #loss divided by number of samples it has trained on

        # Average loss for the epoch
        epoch_train_loss.append(epoch_loss/len(xTrain))

        # Validation step (if provided)
        if xVal is not None and yVal is not None:
            model.eval()  # Set model to evaluation mode
            val_loss = 0.0
            with torch.no_grad():  # No need to track gradients during validation
                val_outputs = model(xVal)
                val_loss = torch.sum(criterion(val_outputs, yVal)).item()

            epoch_val_loss.append(val_loss/len(xVal))
            print(f'Epoch {epoch+1}/{epochs}, Training Loss: {epoch_loss/len(xTrain):.4f}, Validation Loss: {val_loss/len(xVal):.4f}')
        else:
            print(f'Epoch {epoch+1}/{epochs}, Training Loss: {epoch_loss/len(xTrain):.4f}')

    if xVal is not None and yVal is not None:
        return epoch_train_loss, epoch_val_loss, train_loss_per_batch
    else:
        return epoch_train_loss, train_loss_per_batch

## Naive Models

### Weighted Average Model (learned weights)
Essentially a linear single layer neural network

In [ ]:
weighted_avg_model = WeightedAverageModel(timesteps=xTr.shape[1], features=xTr.shape[2])
# Train the Weighted Average Model and capture the loss history
print("Training the Weighted Average Model...")
weighted_avg_train_loss, weighted_avg_val_loss, weighted_avg_nSample_loss = train(
    model=weighted_avg_model,
    xTrain=xTr,
    yTrain=yTr,
    xVal=xTe,  # Validation data
    yVal=yTe,  # Validation labels
    epochs=50,          
    batch_size=xTr.shape[0],      
    lr=0.001,
    shuffle=False        
)

### Neural Network Average Model (learned weights)

In [ ]:
NN_model = MultiStepFullyConnectedNN(timesteps=xTr.shape[1], features=xTr.shape[2], hidden_units=64, forecast_length=yTr.shape[1])

print("Training the Neural Network Model...")
NN_train_loss, NN_val_loss, NN_nSample_loss = train(
    model=NN_model,
    xTrain=xTr,
    yTrain=yTr,
    xVal=xTe,  # Validation data
    yVal=yTe,
    epochs=50,          
    batch_size=10,#len(xTr),      
    lr=0.001,
    shuffle=False        
)

### Basic LSTM Model

In [ ]:
lstm_model = LSTMMultivariate(timesteps=xTr.shape[1], features=xTr.shape[2], lstm_units=64, forecast_length=yTr.shape[1]) #basic LSTM
print("Training the LSTM Model...")
lstm_train_loss, lstm_val_loss, lstm_nSample_loss = train(
    model=lstm_model,
    xTrain=xTr,
    yTrain=yTr,
    xVal=xTe,  # Validation data
    yVal=yTe,
    epochs=50,          
    batch_size=len(xTr),  
    lr=0.001,
    shuffle=False        
)

## Training and Validation Loss Curves

In [ ]:
plt.close(4)

loss_curves_fig = plt.figure(figsize=(10, 6))
loss_curves_axes = loss_curves_fig.add_subplot(1,1,1)
loss_curves_fig.suptitle(f'MSE Loss vs Epochs')

losses = {'weighted_avg_model': (weighted_avg_train_loss,weighted_avg_val_loss), 
          'neural_net_model': (NN_train_loss,NN_val_loss),
          'lstm_model': (lstm_train_loss,lstm_val_loss)}
cmap = plt.get_cmap('hsv')
for i, modelLoss in enumerate(losses.items()):
    modelName = modelLoss[0]
    trainLoss, valLoss = modelLoss[1]
    loss_curves_axes.plot(trainLoss, color=cmap(i/len(losses)), label=f'{modelName} train loss')
    loss_curves_axes.plot(valLoss, color=cmap(i/len(losses)), label=f'{modelName} val loss', linestyle='--')

plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()


## Predictions

In [ ]:
plt.close(5)
#input is a single timeseries of size (length,7)
nFeatures = 7
x = xTr[nSample]
y = yTr_raw[nSample]
y_mean=np.broadcast_to(yTr[nSample],(h,nFeatures))
t = np.arange(0,0.1*(p+h),0.1)[:(p+h)] #added [:np.shape(data)[0]] due to floating point error (e.g. 3102*0.1 == 310.2000005)


feature_data_fig, feature_data_axes = plt.subplots(nFeatures, 1, figsize=(10, 1 * nFeatures), sharex=True, sharey = True)
feature_data_fig.suptitle(f'Features vs Time, file {nSample}')

for iFeature in range(nFeatures):
    feature_data_axes[iFeature].plot(t[:p], x[:, iFeature], label='x', marker = 'o', linestyle = '-', markersize=3, color='grey')
    feature_data_axes[iFeature].plot(t[p:p+h], y[:, iFeature], label='Actual Emotion', marker = 'o', linestyle = '-', markersize=3, color='black')
    feature_data_axes[iFeature].plot(t[p:p+h], y_mean[:, iFeature], label='y', linestyle = '-', linewidth=3, color='green')

models = {'weighted_avg_model': weighted_avg_model,
          'NN_model': NN_model,
          'lstm_model': lstm_model}

for i, modelName in enumerate(models.keys()):
    model = models[modelName]
    model.eval()
    with torch.no_grad():
        yPred = model(torch.from_numpy(xTr[[nSample]]).float()).detach().numpy()
    yPred = np.broadcast_to(yPred.squeeze(), (h,nFeatures))
    for iFeature in range(nFeatures):
        feature_data_axes[iFeature].plot(t[p:p+h], yPred[:, iFeature], label=f'{modelName} Prediction', marker = 'o', linestyle = '-', markersize=1, color=cmap(i/len(models)))

#label subplots
for i in range(nFeatures):       
    feature_data_axes[i].set_ylabel(f'{emotionMap[i]}')
    feature_data_axes[i].grid(True)

feature_data_axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.subplots_adjust(top=0.95)  # Adjust title position
feature_data_axes[-1].set_xlim(0, (p+h)/10)
feature_data_axes[-1].set_ylim(0, 1)
plt.legend()
plt.show()